# Hugging Face Pipelines: Simplifying NLP Code

Hugging Face Transformers provides **pipelines** as high-level APIs that abstract away complex code for common NLP tasks. This reduces the amount of boilerplate code needed, making it easier to perform tasks like sentiment analysis, text generation, and more. Ideal with quick iterations and could even be used in production. It comes with built-in optimizations (handles device placement, batching, etc.). Pipelines handle: Model loading, Tokenization, Inference, and Post-processing. For more details, visit the [Hugging Face Pipelines Docs](https://huggingface.co/docs/transformers/main_classes/pipelines).

**Note**: The manual implementations shown below are simplified to focus on core logic. A full manual equivalent that replicates all pipeline features (device management, batching, error handling, caching, etc.) would require significantly more code—often 2-3 times longer.

In [ ]:
# getting environment variables from .env file
from dotenv import load_dotenv

load_dotenv(override=True)

## Example: Sentiment Analysis
Let's see how pipelines reduce code compared to manual implementation.

In [ ]:
# Install transformers if not already installed
# !pip install transformers

from transformers import pipeline

# Simple pipeline usage
classifier = pipeline("sentiment-analysis")
result = classifier("I love using Hugging Face!")
print(result)

**Manual Implementation (for comparison)**
- Load the model and tokenizer
- Preprocess the text
- Run inference
- Decode and format output

In [ ]:
# Manual way (equivalent to above)
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")

# Preprocess
inputs = tokenizer("I love using Hugging Face!", return_tensors="pt")

# Inference
with torch.no_grad():
    logits = model(**inputs).logits

# Post-process
predicted_class_id = logits.argmax().item()
predicted_label = model.config.id2label[predicted_class_id]
score = torch.softmax(logits, dim=1)[0][predicted_class_id].item()

print([{"label": predicted_label, "score": score}])

## Text Generation Pipeline

Pipelines can generate text based on a prompt. Here's an example using GPT-2.

In [ ]:
from transformers import pipeline

# Text generation
generator = pipeline("text-generation")
result = generator("In a world where AI", max_length=20, num_return_sequences=1)
print(result)

**Manual Implementation (for comparison)**
- Load the model and tokenizer for causal language modeling
- Preprocess the input prompt into tokens
- Generate continuation tokens using the model
- Decode the generated tokens back to text

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

# Load model and tokenizer for text generation
tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2")

# Preprocess
inputs = tokenizer("In a world where AI", return_tensors="pt")

# Generate
outputs = model.generate(**inputs, max_length=20, num_return_sequences=1)

# Decode
result = tokenizer.decode(outputs[0], skip_special_tokens=True)
print([{"generated_text": result}])

## Question Answering Pipeline

This pipeline answers questions based on provided context.

In [ ]:
from transformers import pipeline

# Question answering
qa = pipeline("question-answering")
result = qa(question="What is the capital of France?", context="France is a country in Europe. Paris is its capital city.")
print(result)

**Manual Implementation (for comparison)**
- Load the question answering model and tokenizer
- Preprocess the question and context into input tokens
- Run inference to get start and end position logits
- Extract the answer span using argmax on logits
- Decode the answer tokens to text

In [ ]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
import torch

# Load model and tokenizer for question answering
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased-distilled-squad")
model = AutoModelForQuestionAnswering.from_pretrained("distilbert-base-uncased-distilled-squad")

# Define question and context
question = "What is the capital of France?"
context = "France is a country in Europe. Paris is its capital city."

# Preprocess
inputs = tokenizer(question, context, return_tensors="pt")

# Inference
with torch.no_grad():
    outputs = model(**inputs)

# Post-process
start_logits = outputs.start_logits
end_logits = outputs.end_logits
start_index = torch.argmax(start_logits)
end_index = torch.argmax(end_logits) + 1
answer_tokens = inputs["input_ids"][0][start_index:end_index]
answer = tokenizer.decode(answer_tokens, skip_special_tokens=True)
score = torch.max(start_logits).item()

print({"answer": answer, "start": start_index.item(), "end": end_index.item(), "score": score})

## Fill-Mask Pipeline

This pipeline predicts masked tokens in text.

In [ ]:
from transformers import pipeline

# Fill mask
fill_mask = pipeline("fill-mask")
result = fill_mask("Hugging Face is a <mask> company.")
print(result)

**Manual Implementation (for comparison)**
- Load the masked language model and tokenizer
- Preprocess the text with mask token into input tensors
- Run inference to get prediction logits for all positions
- Identify the mask position and extract its logits
- Get top-k predictions and format results with scores

In [ ]:
from transformers import AutoTokenizer, AutoModelForMaskedLM
import torch

# Load model and tokenizer for fill-mask
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
model = AutoModelForMaskedLM.from_pretrained("distilbert-base-uncased")

# Preprocess
inputs = tokenizer("Hugging Face is a [MASK] company.", return_tensors="pt")

# Inference
with torch.no_grad():
    logits = model(**inputs).logits

# Find mask token index
mask_token_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]

# Get logits for mask
mask_token_logits = logits[0, mask_token_index, :]

# Get top 5 predictions
top_5 = torch.topk(mask_token_logits, 5, dim=1)
scores = top_5.values[0]
tokens = top_5.indices[0]

# Format output similar to pipeline
results = []
for score, token in zip(scores, tokens):
    token_str = tokenizer.decode(token)
    sequence = tokenizer.decode(inputs["input_ids"][0]).replace("[MASK]", token_str)
    results.append({"score": score.item(), "token": token.item(), "token_str": token_str, "sequence": sequence})

print(results)

## Token Classification (Named Entity Recognition) Pipeline

This pipeline identifies entities like persons, organizations, and locations in text.

In [ ]:
from transformers import pipeline

# Named Entity Recognition
ner = pipeline("ner", aggregation_strategy="simple")
result = ner("My name is John Doe and I work at Google in New York.")
print(result)

**Manual Implementation (for comparison)**
- Load the NER model and tokenizer
- Preprocess the text
- Run inference
- Post-process to extract entities

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch

# Load model and tokenizer for NER
tokenizer = AutoTokenizer.from_pretrained("dbmdz/bert-large-cased-finetuned-conll03-english")
model = AutoModelForTokenClassification.from_pretrained("dbmdz/bert-large-cased-finetuned-conll03-english")

# Preprocess
inputs = tokenizer("My name is John Doe and I work at Google in New York.", return_tensors="pt")

# Inference
with torch.no_grad():
    outputs = model(**inputs)

# Simplified post-processing (real NER requires proper entity grouping)
predictions = torch.argmax(outputs.logits, dim=2)
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
labels = [model.config.id2label[pred.item()] for pred in predictions[0]]

# Print token-label pairs (simplified)
entities = []
for token, label in zip(tokens, labels):
    if label != 'O':
        entities.append({"word": token, "entity": label})

print(entities)

## Translator pipeline

In [ ]:
# Downgrade transformers to 4.57.6 to avoid translation pipeline issues
# For now, use the manual implementation below.

from transformers import pipeline

translator = pipeline("translation", src_lang="en", tgt_lang="fr")
result = translator("Hello, how are you?")
print(result)

**Manual Implementation**
- Load the translation model and tokenizer
- Preprocess the text
- Run inference
- Decode and format output

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load model and tokenizer for translation
tokenizer = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-fr")
model = AutoModelForSeq2SeqLM.from_pretrained("Helsinki-NLP/opus-mt-en-fr")

# Preprocess
inputs = tokenizer("Hello, how are you?", return_tensors="pt")

# Generate translation
outputs = model.generate(**inputs, max_length=40, num_beams=4, early_stopping=True)

# Decode
translated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print([{"translation_text": translated_text}])

## Summarization Pipeline

This pipeline summarizes long texts into shorter versions.

In [ ]:
# Downgrade transformers to 4.57.6 to avoid summarization pipeline issues
# For now, use the manual implementation below.

from transformers import pipeline

summarizer = pipeline("summarization")
result = summarizer("Hugging Face is a company that provides tools for natural language processing. They offer pre-trained models and easy-to-use APIs for tasks like text classification, generation, and more. Their transformers library is widely used in the AI community.")
print(result)

**Manual Implementation (for comparison)**
- Load the summarization model and tokenizer
- Preprocess the text
- Run inference
- Decode and format output

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load model and tokenizer for summarization
tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")
model = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-large-cnn")

# Preprocess
inputs = tokenizer("Hugging Face is a company that provides tools for natural language processing. They offer pre-trained models and easy-to-use APIs for tasks like text classification, generation, and more. Their transformers library is widely used in the AI community.", return_tensors="pt", max_length=1024, truncation=True)

# Generate summary
outputs = model.generate(**inputs, max_length=50, num_beams=4, early_stopping=True)

# Decode
summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
print([{"summary_text": summary}])

## Translation Example: English to Swedish

For translation, pipelines can be used, but here's a manual implementation to show the process. (Note: Pipelines for translation require specifying the model and may need task-specific setup.)

In [ ]:
from transformers import MarianTokenizer, MarianMTModel

# Load model and tokenizer for English to Swedish
tokenizer = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-sv")
model = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-en-sv")

# Preprocess
inputs = tokenizer("Hello, how are you?", return_tensors="pt")

# Generate translation
outputs = model.generate(**inputs)

# Decode
result = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(result)